## 🎯 Learning Objectives
* Understand the core concept and motivation behind Corrective RAG (CRAG).
* Learn how self-grading mechanisms enhance retrieval quality in RAG systems.
* Implement a basic CRAG workflow using LangGraph for conditional re-retrieval.
* Analyze the trade-offs and practical applications of CRAG in advanced RAG architectures.


## Lesson ADV04-L03: Corrective RAG (CRAG): Self-Grading and Re-Retrieval

Welcome to Lesson ADV04-L03, where we dive into **Corrective RAG (CRAG)**, a powerful technique designed to significantly enhance the reliability and accuracy of Retrieval-Augmented Generation (RAG) systems. As AI search engineers, you're constantly battling issues like hallucinations, irrelevant context, and outdated information. CRAG offers a robust solution by introducing a self-correction mechanism into the retrieval process.

### The Problem: Flawed Retrieval

Traditional RAG systems follow a straightforward path: retrieve documents based on a query, then generate an answer using those documents. The critical vulnerability here is the quality of the initial retrieval. If the retrieved documents are irrelevant, incomplete, or misleading, even the most sophisticated Large Language Model (LLM) will struggle to produce an accurate answer, often leading to hallucinations or nonsensical responses.

Imagine a student preparing for an exam. They look up information in a textbook. If they accidentally pick up the wrong textbook or find an irrelevant chapter, no matter how good they are at synthesizing information, their answer will be incorrect. Similarly, a RAG system relying on poor retrieval is fundamentally flawed.

### Introducing Corrective RAG (CRAG)

CRAG addresses this by adding an intelligent **self-grading** step. After the initial retrieval, an LLM (often called a "retrieval grader" or "critic") assesses the relevance and quality of the retrieved documents *in the context of the user's query*. If the grader determines the documents are insufficient or irrelevant, the system doesn't just proceed to generation. Instead, it triggers a **re-retrieval** strategy, attempting to find better information.

Think of it like a meticulous fact-checker. Before publishing an article, a fact-checker reviews the sources. If the sources are weak or don't support the claims, they don't just let it pass; they demand better, more reliable sources. CRAG applies this principle to RAG.

### How CRAG Works: A Step-by-Step Flow

1.  **Initial Retrieval**: The RAG system performs a standard retrieval operation, fetching a set of documents from its knowledge base based on the user's query.

2.  **Retrieval Assessment (Self-Grading)**: An LLM, specifically designed as a "retrieval grader," evaluates the retrieved documents against the original query. It determines if the documents are relevant and sufficient to answer the query accurately. This assessment typically results in a binary decision: "relevant" or "irrelevant" (or a confidence score).

3.  **Conditional Re-Retrieval/Augmentation**: 
    *   **If "Relevant"**: The system proceeds with the initially retrieved documents to the generation phase.
    *   **If "Irrelevant"**: The system initiates a corrective action. This could involve:
        *   **Re-ranking**: Re-ordering the existing retrieved documents.
        *   **Query Expansion/Rewriting**: Modifying the original query to be more precise or broader, then re-running the retriever.
        *   **Alternative Retriever**: Using a different retrieval mechanism (e.g., keyword search instead of vector search, or a specialized search tool like a web search engine).
        *   **Augmentation**: Fetching additional information from external sources (e.g., web search) to supplement the existing context.

4.  **Generation**: With the (potentially improved) set of documents, the main LLM generates the final answer to the user's query.

### Key Components of a CRAG System

*   **Retriever**: The component responsible for fetching documents from the knowledge base.
*   **Retrieval Grader (LLM)**: A specialized LLM that evaluates the relevance of retrieved documents to the query.
*   **Re-retrieval Strategy**: The logic and tools employed when the initial retrieval is deemed insufficient (e.g., another retriever, web search tool).
*   **Generator (LLM)**: The main LLM that synthesizes the final answer based on the refined context.

CRAG is particularly valuable in high-stakes applications where accuracy is paramount, such as legal research, medical diagnostics, financial analysis, and critical enterprise knowledge management. By adding this intelligent feedback loop, CRAG significantly elevates the trustworthiness and performance of RAG systems, pushing them closer to human-level fact-checking capabilities.


In [ ]:
import os
from typing import List, Dict, Any, TypedDict

# Ensure you have LangChain, LangGraph, and Ollama installed:
# pip install langchain langchain-community langgraph beautifulsoup4

# For local LLM inference, ensure Ollama is running and you have models pulled.
# Example: ollama run llama3, ollama run nomic-embed-text

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, END

# --- Configuration and Model Setup ---

# Initialize Ollama LLM and Embeddings. 
# Make sure Ollama server is running and models are pulled (e.g., llama3, nomic-embed-text).
llm = Ollama(model="llama3", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# --- Data Ingestion and Vector Store Setup ---

# Load a sample document (e.g., from a web page)
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-rag/",
    "https://lilianweng.github.io/posts/2024-02-05-crag/" # Specifically for CRAG
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs_list)

# Create a vector store (using Chroma for simplicity)
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- LangGraph State Definition ---

# Define the state for our LangGraph workflow
class GraphState(TypedDict):
    question: str
    documents: List[str] # Retrieved documents
    web_search_documents: List[str] # Documents from web search (if re-retrieval occurs)
    retrieval_grade: str # "yes" or "no" based on grader's assessment
    generation_output: str # Final generated answer

# --- LangGraph Nodes and Functions ---

def retrieve(state: GraphState) -> GraphState:
    """Initial document retrieval from our vector store."""
    print("---NODE: RETRIEVE DOCUMENTS---")
    question = state["question"]
    documents = retriever.invoke(question)
    # Convert Document objects to string for simpler state management
    doc_contents = [doc.page_content for doc in documents]
    return {"documents": doc_contents, "question": question}


def grade_documents(state: GraphState) -> GraphState:
    """Grades the retrieved documents for relevance to the question.
       If irrelevant, triggers re-retrieval (simulated by web search)."""
    print("---NODE: GRADE DOCUMENTS---")
    question = state["question"]
    documents = state["documents"]

    # Prompt for the retrieval grader LLM
    grade_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a grader AI. Your task is to assess the relevance of the provided documents to the user's question. "
                   "Respond with 'yes' if the documents are relevant and sufficient to answer the question, otherwise respond with 'no'. "
                   "Do not provide any other explanation."),
        ("human", "Question: {question}\n\nDocuments:\n{documents}\n\nIs this relevant? (yes/no)")
    ])

    retrieval_grader = grade_prompt | llm | StrOutputParser()
    
    # Concatenate documents for the grader
    docs_for_grader = "\n\n".join(documents)
    
    grade = retrieval_grader.invoke({"question": question, "documents": docs_for_grader}).strip().lower()
    print(f"Retrieval Grade: {grade}")

    return {"question": question, "documents": documents, "retrieval_grade": grade}


def web_search(state: GraphState) -> GraphState:
    """Performs a simulated web search for additional context.
       In a real system, this would use a tool like Tavily, Google Search API, etc."""
    print("---NODE: PERFORM WEB SEARCH (Re-retrieval)---")
    question = state["question"]
    
    # Simulate web search results. In a real scenario, this would call an external API.
    # For demonstration, we'll just add a generic relevant document.
    simulated_web_results = [
        f"Web search result for '{question}': Corrective RAG (CRAG) is an advanced RAG technique that involves a retrieval grader to assess initial retrieval quality. If documents are deemed irrelevant, it triggers re-retrieval, often using web search or alternative strategies. This improves factual consistency and reduces hallucinations. Key components include a retriever, a grader LLM, and a re-retrieval mechanism. It's crucial for high-stakes applications."
    ]
    
    # Combine existing documents with web search results
    combined_documents = state["documents"] + simulated_web_results
    
    return {"question": question, "documents": combined_documents, "web_search_documents": simulated_web_results}


def generate(state: GraphState) -> GraphState:
    """Generates the final answer based on the provided documents."""
    print("---NODE: GENERATE ANSWER---")
    question = state["question"]
    documents = state["documents"]

    # Prompt for the answer generation LLM
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert assistant for question-answering tasks. Use the following context to answer the question. "
                   "If you don't know the answer, state that you don't know, but try to be helpful and provide relevant information from the context."
                   "Do not make up information."),
        ("human", "Question: {question}\n\nContext:\n{context}\n\nAnswer:")
    ])

    # Format documents for the context
    context = "\n\n".join(documents)
    
    rag_chain = rag_prompt | llm | StrOutputParser()
    
    answer = rag_chain.invoke({"question": question, "context": context})
    print(f"Generated Answer: {answer}")
    
    return {"question": question, "documents": documents, "generation_output": answer}

# --- LangGraph Workflow Definition ---

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("web_search", web_search)
workflow.add_node("generate", generate)

# Set entry point
workflow.set_entry_point("retrieve")

# Add edges
workflow.add_edge("retrieve", "grade_documents")

# Conditional edge from grade_documents
def route_documents(state: GraphState) -> str:
    """Routes based on the retrieval grade."""
    if state["retrieval_grade"] == "yes":
        print("---ROUTE: DOCUMENTS ARE RELEVANT, PROCEED TO GENERATION---")
        return "generate"
    else:
        print("---ROUTE: DOCUMENTS ARE IRRELEVANT, PERFORM WEB SEARCH---")
        return "web_search"

workflow.add_conditional_edges(
    "grade_documents",
    route_documents,
    {
        "generate": "generate",
        "web_search": "web_search"
    }
)

workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()

# --- Run the CRAG Workflow ---

# Example 1: Query likely to be answered by initial retrieval
print("\n--- RUNNING CRAG WORKFLOW (EXAMPLE 1: GOOD RETRIEVAL) ---")
question_1 = "What is Corrective RAG (CRAG) and why is it used?"
inputs_1 = {"question": question_1}
for s in app.stream(inputs_1):
    print(s)

# Example 2: Query that might require re-retrieval (simulated by web search)
# This query is intentionally designed to be slightly outside the initial docs' direct focus
# to trigger the 'no' grade and web search.
print("\n--- RUNNING CRAG WORKFLOW (EXAMPLE 2: POOR RETRIEVAL, TRIGGERS WEB SEARCH) ---")
question_2 = "Explain the role of a retrieval grader in improving RAG systems."
inputs_2 = {"question": question_2}
for s in app.stream(inputs_2):
    print(s)

# Example 3: A more general question that might benefit from web search if initial docs are too specific
print("\n--- RUNNING CRAG WORKFLOW (EXAMPLE 3: ANOTHER POOR RETRIEVAL SCENARIO) ---")
question_3 = "What are the main challenges in building advanced RAG systems?"
inputs_3 = {"question": question_3}
for s in app.stream(inputs_3):
    print(s)


### Interpreting the Code Output and Performance Considerations

The provided code demonstrates a basic, yet functional, Corrective RAG (CRAG) system using LangGraph. Let's break down what you'll observe in the output and discuss its implications:

#### Interpreting the Output

1.  **Node Execution Trace**: You'll see `---NODE: ...---` messages indicating which part of the LangGraph workflow is currently active. This clearly shows the flow of execution.
2.  **`retrieve`**: This node fetches initial documents from our `Chroma` vector store based on the `question`.
3.  **`grade_documents`**: This is the core of CRAG. The `retrieval_grader` LLM evaluates the relevance of the retrieved documents. You'll see `Retrieval Grade: yes` or `Retrieval Grade: no`.
    *   **If `yes`**: The `route_documents` function directs the flow directly to the `generate` node. This means the initial retrieval was deemed sufficient.
    *   **If `no`**: The `route_documents` function directs the flow to the `web_search` node. This signifies that the initial retrieval was inadequate, and a re-retrieval strategy is being employed.
4.  **`web_search`**: This node simulates fetching additional, broader information (e.g., from the internet) when the initial retrieval fails. In a real-world scenario, this would involve calling a search API (like Tavily, Google Search, etc.). The documents from this step are then combined with any previously retrieved documents.
5.  **`generate`**: Finally, the `generate` node takes all available relevant documents (either initial or augmented) and the original question to produce the final answer. You'll see `Generated Answer: ...`.

By observing these steps, you can clearly see how CRAG dynamically adapts its retrieval strategy based on the quality assessment, leading to more robust and accurate answers, especially for complex or nuanced queries.

#### Performance Trade-offs

While CRAG significantly boosts accuracy, it's crucial to understand its performance implications:

1.  **Increased Latency**: The most apparent trade-off is increased latency. A CRAG system involves additional LLM calls (for grading) and potentially extra retrieval steps (re-retrieval/web search). This means each query takes longer to process compared to a simple RAG system.
2.  **Computational Cost**: More LLM inferences and potentially more complex retrieval operations translate to higher computational costs, especially if using expensive proprietary LLMs or external search APIs.
3.  **Complexity**: The LangGraph implementation, while powerful, adds a layer of complexity to the system design and debugging compared to a linear RAG chain.

#### Typical Use Cases

CRAG is not always necessary for every RAG application. It shines in scenarios where:

*   **High Accuracy is Critical**: Applications in legal, medical, financial, or scientific domains where factual errors can have severe consequences.
*   **Knowledge Base Volatility**: When the underlying knowledge base is constantly changing, or queries might require very up-to-date information not present in the static vector store.
*   **Ambiguous or Complex Queries**: For questions that are open to interpretation or require synthesizing information from multiple, potentially disparate sources.
*   **Reducing Hallucinations**: When the primary goal is to minimize the LLM's tendency to generate incorrect or fabricated information by ensuring the context is always highly relevant and comprehensive.
*   **Enterprise Search**: For internal knowledge management systems where users expect highly reliable answers from a vast and diverse set of internal documents and potentially external web sources.

By strategically applying CRAG, AI search engineers can build RAG systems that are not only powerful but also trustworthy and resilient to common retrieval failures.


### Resources

*   **LangGraph Documentation**: The official documentation for building stateful, multi-actor applications with LLMs: [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **Corrective RAG (CRAG) Paper**: Explore the original research paper that introduced CRAG: [https://arxiv.org/abs/2401.15884](https://arxiv.org/abs/2401.15884)
*   **Lilian Weng's Blog Post on Advanced RAG**: A comprehensive overview of advanced RAG techniques, including CRAG: [https://lilianweng.github.io/posts/2023-10-25-adv-rag/](https://lilianweng.github.io/posts/2023-10-25-adv-rag/)
*   **LangChain RAG Tutorials**: General resources and examples for building RAG systems with LangChain: [https://python.langchain.com/docs/use_cases/question_answering/](https://python.langchain.com/docs/use_cases/question_answering/)
*   **Ollama**: For running open-source LLMs locally, which is crucial for 2026-ready, privacy-preserving, and cost-effective RAG systems: [https://ollama.com/](https://ollama.com/)
